# ML Experiment 6: Tic-Tac-Toe using Reinforcement Learning (Q-learning)
Tasks: (a) set up environment, (b) define the game, (c) build the RL model, (d) train, (e) test.

*Note:* the lab-manual code chose `argmax` over the Q-values of available moves, which returns a position in the list rather than the board square; this is fixed here (`available_moves[idx]`). Learning rate is also lowered to a more standard value so training converges.

## a) Environment setup & b) Tic-Tac-Toe game

In [ ]:
import numpy as np

class TicTacToeEnvironment:
    def __init__(self):
        self.state = [0] * 9
        self.is_terminal = False

    def reset(self):
        self.state = [0] * 9
        self.is_terminal = False

    def get_available_moves(self):
        return [i for i, mark in enumerate(self.state) if mark == 0]

    def make_move(self, move, player_mark):
        self.state[move] = player_mark

    def check_win(self, player_mark):
        winning_states = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],   # rows
            [0, 3, 6], [1, 4, 7], [2, 5, 8],   # columns
            [0, 4, 8], [2, 4, 6]               # diagonals
        ]
        for idx in winning_states:
            if all(self.state[i] == player_mark for i in idx):
                self.is_terminal = True
                return True
        return False

    def is_draw(self):
        return 0 not in self.state

## c) Q-learning agent

In [ ]:
class QLearningAgent:
    def __init__(self, learning_rate=0.5, discount_factor=0.9, exploration_rate=0.3):
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.exploration_rate = exploration_rate
        self.q_table = np.zeros((3**9, 9))

    def get_state_index(self, state):
        idx = 0
        for i, mark in enumerate(state):
            idx += (3 ** i) * (mark + 1)
        return idx

    def choose_action(self, state, available_moves):
        s = self.get_state_index(state)
        if np.random.random() < self.exploration_rate:
            return int(np.random.choice(available_moves))
        q = self.q_table[s, available_moves]
        return int(available_moves[int(np.argmax(q))])

    def update_q_table(self, state, action, next_state, reward):
        s = self.get_state_index(state)
        max_q = np.max(self.q_table[self.get_state_index(next_state)]) if next_state is not None else 0
        self.q_table[s, action] = (1 - self.learning_rate) * self.q_table[s, action] + \
                                  self.learning_rate * (reward + self.discount_factor * max_q)

## d) Training (self-play)
The mover gets +10 for a win, 0 for a draw, and the opponent's previous move is penalised -10 when the opponent wins.

In [ ]:
def evaluate_agents(agent1, agent2, num_episodes=1000):
    env = TicTacToeEnvironment()
    agent1_wins = agent2_wins = draws = 0
    for _ in range(num_episodes):
        env.reset()
        current = agent1
        last = {agent1: None, agent2: None}      # (state, action) of each agent's previous move
        while not env.is_terminal:
            moves = env.get_available_moves()
            state = env.state.copy()
            action = current.choose_action(state, moves)
            mark = 1 if current is agent1 else -1
            env.make_move(action, mark)
            other = agent2 if current is agent1 else agent1

            if env.check_win(mark):
                current.update_q_table(state, action, None, 10)
                if last[other]:                    # opponent's last move led to a loss
                    s, a = last[other]
                    other.update_q_table(s, a, None, -10)
                if current is agent1: agent1_wins += 1
                else: agent2_wins += 1
                break
            elif env.is_draw():
                current.update_q_table(state, action, None, 0)
                draws += 1
                break
            current.update_q_table(state, action, env.state.copy(), 0)
            last[current] = (state, action)
            current = other
    return agent1_wins, agent2_wins, draws

agent1 = QLearningAgent()
agent2 = QLearningAgent()
w1, w2, d = evaluate_agents(agent1, agent2, num_episodes=1000)
print(f"Agent 1 wins: {w1}\nAgent 2 wins: {w2}\nDraws: {d}")

In [ ]:
# Train longer; watch draws increase as both agents improve
for r in range(5):
    w1, w2, d = evaluate_agents(agent1, agent2, num_episodes=10000)
    print(f"Round {r+1}: A1 wins={w1}, A2 wins={w2}, draws={d}")

## e) Testing: trained agent vs a random player

In [ ]:
def play_vs_random(agent, games=1000, agent_first=True):
    env = TicTacToeEnvironment()
    old = agent.exploration_rate; agent.exploration_rate = 0
    win = loss = draw = 0
    for _ in range(games):
        env.reset()
        turn_agent = agent_first
        while True:
            moves = env.get_available_moves()
            if turn_agent:
                m = agent.choose_action(env.state.copy(), moves); mark = 1
            else:
                m = int(np.random.choice(moves)); mark = -1
            env.make_move(m, mark)
            if env.check_win(mark):
                if turn_agent: win += 1
                else: loss += 1
                break
            if env.is_draw():
                draw += 1; break
            turn_agent = not turn_agent
    agent.exploration_rate = old
    return win, loss, draw

print("Agent1 (moves first) vs random  [win, loss, draw]:", play_vs_random(agent1, 1000, True))
print("Agent2 (moves second) vs random [win, loss, draw]:", play_vs_random(agent2, 1000, False))

## Play against the trained agent (optional)
You are **O**, the agent is **X** and moves first. Enter a square 0-8.

In [ ]:
def show(s):
    sym = {1:'X', -1:'O', 0:'.'}
    for r in range(3): print(' '.join(sym[s[r*3+c]] for c in range(3)))
    print()

def human_game(agent):
    env = TicTacToeEnvironment()
    old = agent.exploration_rate; agent.exploration_rate = 0
    turn_agent = True
    while True:
        moves = env.get_available_moves()
        if turn_agent:
            m = agent.choose_action(env.state.copy(), moves); mark = 1
        else:
            show(env.state)
            m = int(input(f"Your move {moves}: ")); mark = -1
            if m not in moves: print("Invalid"); continue
        env.make_move(m, mark)
        if env.check_win(mark):
            show(env.state); print("Agent wins!" if turn_agent else "You win!"); break
        if env.is_draw():
            show(env.state); print("Draw!"); break
        turn_agent = not turn_agent
    agent.exploration_rate = old

# human_game(agent1)   # uncomment to play